## Runtime / Drive bootstrap

In [ ]:
from pathlib import Path
import base64, io, os, shutil, sys, zipfile
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/Project_Paper01_SCAL_DAS')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
PACKAGE_ZIP = PROJECT_ROOT / 'Paper01_PHASE2_v0.1_PACKAGE.zip'
EMBEDDED_PACKAGE_B64 = '...'
WORK_PACKAGE_ROOT = Path('/content/phase2_package')
shutil.rmtree(WORK_PACKAGE_ROOT, ignore_errors=True)
WORK_PACKAGE_ROOT.mkdir(parents=True, exist_ok=True)
if PACKAGE_ZIP.is_file():
    with zipfile.ZipFile(PACKAGE_ZIP, 'r') as zf: zf.extractall(WORK_PACKAGE_ROOT)
    BOOTSTRAP_SOURCE = str(PACKAGE_ZIP)
else:
    payload = base64.b64decode(EMBEDDED_PACKAGE_B64)
    with zipfile.ZipFile(io.BytesIO(payload), 'r') as zf: zf.extractall(WORK_PACKAGE_ROOT)
    BOOTSTRAP_SOURCE = 'embedded fallback'
candidates = [p for p in WORK_PACKAGE_ROOT.rglob('phase2_compact') if p.is_dir()]
if len(candidates) != 1: raise RuntimeError(f'Expected one phase2_compact package, found {candidates}')
package_root = candidates[0].parent
if str(package_root) not in sys.path: sys.path.insert(0, str(package_root))
print('Phase-2 bootstrap:', BOOTSTRAP_SOURCE)
print('package_root:', package_root)


## Configuration

In [ ]:
import shutil, zipfile
from phase2_compact.config import Phase2Config
from phase2_compact.pipeline import Phase2Paths, run_phase2, run_dry_run

EXPECTED_PHASE1_SHA = 'd5248a7d5730e7ad6857c5475f248e27cd315d9c5e26235eaba57421936cfe44'
cfg = Phase2Config()
PHASE1_ROOT = PROJECT_ROOT / 'Phase1_Benchmark_v0.1_FROZEN'
if not (PHASE1_ROOT / 'BENCHMARK_SHA256.txt').is_file():
    zips = sorted(PROJECT_ROOT.glob('Phase1_Benchmark*v0.1*FROZEN*.zip'))
    if len(zips) != 1:
        raise FileNotFoundError(f'Place exactly one final Phase-1 frozen ZIP in {{PROJECT_ROOT}}; found {{zips}}')
    PHASE1_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zips[0], 'r') as zf: zf.extractall(PHASE1_ROOT)
RAW_ROOT = PROJECT_ROOT / 'Raw_DAS'  # Change only if your original raw HDF5 recordings are elsewhere.
OUTPUT_ROOT = PROJECT_ROOT / 'Phase2_Compact_v0.1'
MANUAL_RAW_MAP = {}  # Optional: {'recording_id': '/absolute/path/to/file.h5'} after explicit review.
GEOMETRY_OVERRIDES = {}  # Optional: {'recording_id': {'window_length': ..., 'hop': ..., 'time_axis': 0}} after explicit review.
paths = Phase2Paths(PHASE1_ROOT, RAW_ROOT, OUTPUT_ROOT)
print('Phase-1:', PHASE1_ROOT)
print('Raw DAS:', RAW_ROOT)
print('Output:', OUTPUT_ROOT)

## Phase-1 frozen benchmark verification

In [ ]:
from phase2_compact.phase1 import load_phase1_contract
contract = load_phase1_contract(paths.phase1_root, cfg)
assert contract.phase1_sha == EXPECTED_PHASE1_SHA
assert len(contract.events) == 137
print('Phase-1 SHA PASS:', contract.phase1_sha)
print('Frozen events:', len(contract.events))

## Raw-dataset discovery

In [ ]:
from phase2_compact.discovery import discover_raw_files
raw_files = discover_raw_files(paths.raw_root)
if not raw_files:
    raise FileNotFoundError(f'No .h5/.hdf5 raw DAS files found under {{paths.raw_root}}')
print('Raw HDF5 files:', len(raw_files))
for p in raw_files[:20]: print(' -', p)

## HDF5 structural audit

In [ ]:
import pandas as pd, json
from phase2_compact.discovery import inspect_hdf5_file
structures = [inspect_hdf5_file(p) for p in raw_files]
structure_table = pd.DataFrame([{
    'file': str(s.path.relative_to(paths.raw_root)), 'dataset_path': s.dataset_path,
    'shape': str(s.shape), 'dtype': s.dtype, 'attributes': json.dumps(s.attributes, sort_keys=True)
} for s in structures])
display(structure_table)
print('Review dataset paths, shapes, dtypes and timing attributes before continuing.')

## Recording mapping

In [ ]:
from phase2_compact.mapping import build_recording_map
recording_map = build_recording_map(contract, structures, manual_map=MANUAL_RAW_MAP)
display(recording_map)
print(recording_map['mapping_status'].value_counts(dropna=False))

## Mapping integrity gate

In [ ]:
from phase2_compact.mapping import validate_recording_map
from phase2_compact.coordinates import resolve_window_geometry
validate_recording_map(recording_map)
structure_by_path = {str(Path(s.path).resolve()): s for s in structures}
geometry_rows=[]
for r in recording_map.itertuples(index=False):
    s = structure_by_path[str(Path(r.source_raw_file).resolve())]
    g = resolve_window_geometry(s, GEOMETRY_OVERRIDES.get(r.recording_id))  # HARD STOP here if window_length/hop/time_axis are not explicit.
    geometry_rows.append({'recording_id':r.recording_id,'window_length':g.window_length,'hop':g.hop,'time_axis':g.time_axis,'source':g.source_method})
geometry_table=pd.DataFrame(geometry_rows)
display(geometry_table)
print('Mapping + raw-coordinate geometry gate: PASS')

## Dry-run extraction on selected events

In [ ]:
DRY_RUN_ROOT = Path('/content/phase2_dry_run')
shutil.rmtree(DRY_RUN_ROOT, ignore_errors=True)
dry_paths = Phase2Paths(paths.phase1_root, paths.raw_root, DRY_RUN_ROOT)
dry_ids = contract.events.sort_values(['class_name','event_id']).groupby('class_name', sort=True).head(1)['event_id'].tolist()[:3]
print('Dry-run event IDs:', dry_ids)
dry_result = run_dry_run(cfg, dry_paths, event_ids=dry_ids, manual_map=MANUAL_RAW_MAP, geometry_overrides=GEOMETRY_OVERRIDES)
print('Dry-run samples:', len(dry_result.sample_registry))

## Dry-run signal diagnostics

In [ ]:
display(dry_result.sample_registry[['sample_id','event_id','class_name','b1_role','model_time','model_channels']].head(20))
for p in sorted((dry_paths.output_root/'figures').glob('*.png')): print('Figure:', p)
print('Dry-run completed without final freeze manifest:', not (dry_paths.output_root/'PHASE2_MANIFEST.json').exists())

## Full deterministic extraction plan

In [ ]:
import math
geom_by_recording = {r['recording_id']: r for r in geometry_rows}
est=[]
for e in contract.events.itertuples(index=False):
    hop=geom_by_recording[e.recording_id]['hop']
    stride=max(1, math.ceil(cfg.model_time/hop))
    estimated=1 + max(0, math.ceil((int(e.window_end)-int(e.window_start))/stride))
    est.append(estimated)
print('Frozen events:', len(contract.events))
print('Estimated model samples:', sum(est))
print('Model patch:', cfg.model_time, 'x', cfg.model_channels)
print('Extraction is shard-resumable; completed verified shards will be skipped.')

## Resume-state inspection

In [ ]:
from phase2_compact.checkpoint import load_checkpoint
state = load_checkpoint(paths.output_root)
print('Checkpoint Phase-1 SHA:', state.phase1_sha or '<none>')
print('Verified completed shards:', len(state.shards))
for k in list(sorted(state.shards))[:20]: print(' -', k)

## Full extraction

In [ ]:
full_result = run_phase2(cfg, paths, manual_map=MANUAL_RAW_MAP, geometry_overrides=GEOMETRY_OVERRIDES)
print('Samples:', len(full_result.sample_registry))
print('Skipped verified shards this run:', full_result.skipped_shards)
print('Phase-2 portable SHA:', full_result.final_sha)

## Sample registry construction

In [ ]:
sample_registry = pd.read_csv(paths.output_root/'metadata'/'SAMPLE_REGISTRY.csv')
print('Samples:', len(sample_registry), 'Events:', sample_registry['event_id'].nunique())
display(pd.crosstab(sample_registry['class_name'], sample_registry['b1_role']))
display(sample_registry.head())

## B1 linkage/leakage audit

In [ ]:
b1_audit = pd.read_csv(paths.output_root/'audits'/'B1_SAMPLE_LEAKAGE_AUDIT.csv')
phase1_link = pd.read_csv(paths.output_root/'audits'/'PHASE1_LINKAGE_AUDIT.csv')
display(phase1_link)
display(b1_audit)
assert phase1_link['passed'].all() and b1_audit['passed'].all()

## B3 linkage/leakage audit

In [ ]:
b3_audit = pd.read_csv(paths.output_root/'audits'/'B3_SAMPLE_LEAKAGE_AUDIT.csv')
display(b3_audit)
assert b3_audit['passed'].all()
print('Primary B3: F4_to_F1; active classes: car + construction')

## Signal-quality audit

In [ ]:
signal_audit = pd.read_csv(paths.output_root/'audits'/'SIGNAL_QUALITY_AUDIT.csv')
signal_ranges = pd.read_csv(paths.output_root/'audits'/'SIGNAL_RANGE_AUDIT.csv')
display(signal_audit)
display(signal_ranges.describe(include='all'))
assert signal_audit['passed'].all()

## Shard integrity audit

In [ ]:
shard_audit = pd.read_csv(paths.output_root/'audits'/'SHARD_INTEGRITY_AUDIT.csv')
display(shard_audit)
assert shard_audit['valid'].all()
print('Shards:', len(shard_audit), 'Samples:', int(shard_audit['sample_count'].sum()))

## Unit/regression tests

In [ ]:
import os, subprocess, sys
# The notebook kernel sys.path is not inherited automatically by subprocess pytest.
env = os.environ.copy()
existing = env.get('PYTHONPATH','')
env['PYTHONPATH'] = str(package_root) if not existing else str(package_root)+os.pathsep+existing
proc = subprocess.run([sys.executable,'-m','pytest','-q','tests'],cwd=str(package_root),env=env,text=True,capture_output=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError('Standalone Phase-2 unit/regression tests failed')
UNIT_TESTS_PASS=True
print('UNIT_TESTS_PASS =', UNIT_TESTS_PASS)

## Final freeze

In [ ]:
final_gates = pd.read_csv(paths.output_root/'audits'/'FINAL_INTEGRITY_GATES.csv')
display(final_gates)
if len(final_gates) != 19 or not final_gates['passed'].all():
    raise RuntimeError('Phase-2 final integrity gates are not all PASS')
print('19/19 mandatory integrity gates PASS')
print('Frozen SHA:', (paths.output_root/'PHASE2_SHA256.txt').read_text().strip())

## Deterministic rebuild / portable-hash verification

In [ ]:
from phase2_compact.freeze import verify_frozen_phase2
portable_sha = verify_frozen_phase2(paths.output_root)
if portable_sha != full_result.final_sha:
    raise RuntimeError(f'Portable SHA mismatch: {{portable_sha}} != {{full_result.final_sha}}')
print('Persisted-byte freeze verification PASS:', portable_sha)
print('Deterministic rebuild gate:', final_gates.loc[final_gates.gate=='deterministic_rebuild_match','passed'].iloc[0])

## Final summary

In [ ]:
summary = {
    'phase1_sha': contract.phase1_sha,
    'phase2_sha': portable_sha,
    'events': int(sample_registry['event_id'].nunique()),
    'samples': int(len(sample_registry)),
    'shards': int(len(shard_audit)),
    'gates_passed': int(final_gates['passed'].sum()),
}
print(summary)
print('Phase-2 FINAL PASS. The frozen output is ready for Phase-3 model experiments.')

## Export ZIP

In [ ]:
import zipfile
export_zip = PROJECT_ROOT / 'Phase2_Compact_v0.1_FROZEN.zip'
with zipfile.ZipFile(export_zip,'w',compression=zipfile.ZIP_STORED) as zf:
    for p in sorted(paths.output_root.rglob('*')):
        if p.is_file(): zf.write(p,p.relative_to(paths.output_root).as_posix())
print('Exported:', export_zip)
print('Size bytes:', export_zip.stat().st_size)